# Bank Mktg - Evaluate Model

Finds the latest training job, creates a SageMaker Model, runs batch
predictions on the test set, computes evaluation metrics (AUC, accuracy, F1),
and logs results to MLflow.

In [ ]:
# Parameters injected by SageMakerNotebookOperator
training_job_prefix = ""
model_prefix = ""
image_uri = ""
execution_role_arn = ""
region = ""
test_data_path = ""
output_path = ""
tracking_server_arn = ""
experiment_name = ""

In [ ]:
import boto3
import json
import os
import time
import numpy as np
from io import StringIO

In [ ]:
# Resolve parameters
region = region or os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')
sm = boto3.client('sagemaker', region_name=region)
s3 = boto3.client('s3', region_name=region)

print(f'Training prefix: {training_job_prefix}')
print(f'Model prefix:    {model_prefix}')
print(f'Test data:       {test_data_path}')
print(f'Output:          {output_path}')
print(f'Region:          {region}')

In [ ]:
# Find latest completed training job
resp = sm.list_training_jobs(SortBy='CreationTime', SortOrder='Descending', MaxResults=20)
matched = [j for j in resp['TrainingJobSummaries']
           if j['TrainingJobName'].startswith(training_job_prefix)
           and j['TrainingJobStatus'] == 'Completed']

assert matched, f'No completed training jobs found with prefix: {training_job_prefix}'
training_job_name = matched[0]['TrainingJobName']

# Get model artifact location
job_info = sm.describe_training_job(TrainingJobName=training_job_name)
model_data_url = job_info['ModelArtifacts']['S3ModelArtifacts']

# Derive model name
suffix = training_job_name[len(training_job_prefix):]
model_name = f'{model_prefix}{suffix}'

print(f'Training job:  {training_job_name}')
print(f'Model name:    {model_name}')
print(f'Model data:    {model_data_url}')

In [ ]:
# Create SageMaker model (delete if exists)
try:
    sm.describe_model(ModelName=model_name)
    print(f'Model {model_name} exists, deleting...')
    sm.delete_model(ModelName=model_name)
except sm.exceptions.ClientError as e:
    if 'Could not find model' in str(e):
        print(f'No existing model, creating fresh')
    else:
        raise

sm.create_model(
    ModelName=model_name,
    PrimaryContainer={'Image': image_uri, 'ModelDataUrl': model_data_url},
    ExecutionRoleArn=execution_role_arn,
)
print(f'Created model: {model_name}')

In [ ]:
# Run batch transform on test data
# First, prepare test data by stripping the label column (col 0)
# XGBoost inference expects features only
from urllib.parse import urlparse

parsed_test = urlparse(test_data_path.rstrip('/'))
test_bucket = parsed_test.netloc
test_prefix = parsed_test.path.lstrip('/').rstrip('/') + '/'

# Use a run-specific inference prefix to avoid stale files from previous runs
inference_prefix = test_prefix.rstrip('/').rsplit('/', 1)[0] + f'/inference/{model_name}/'

# Clean any existing files in this run-specific inference prefix
existing = s3.list_objects_v2(Bucket=test_bucket, Prefix=inference_prefix)
for obj in existing.get('Contents', []):
    s3.delete_object(Bucket=test_bucket, Key=obj['Key'])

# List test files (only directly under test/)
test_objects = s3.list_objects_v2(Bucket=test_bucket, Prefix=test_prefix)
inference_files_written = []

for obj in test_objects.get('Contents', []):
    key = obj['Key']
    if key.endswith('.csv') and key.startswith(test_prefix) and '/' not in key[len(test_prefix):]:
        body = s3.get_object(Bucket=test_bucket, Key=key)['Body'].read().decode('utf-8')
        # Strip first column (label) from each row
        features_only = '\n'.join(','.join(line.split(',')[1:]) for line in body.strip().split('\n'))
        dest_key = inference_prefix + key.split('/')[-1]
        s3.put_object(Bucket=test_bucket, Key=dest_key, Body=features_only, ContentType='text/csv')
        inference_files_written.append(dest_key)
        print(f'Prepared inference data: s3://{test_bucket}/{dest_key}')

inference_data_path = f's3://{test_bucket}/{inference_prefix}'

# Use a run-specific output path for batch transform results
eval_output_prefix = output_path.rstrip('/') + f'/{model_name}/'

transform_job_name = f'bank-mktg-eval{suffix}'

print(f'Starting batch transform: {transform_job_name}')
print(f'Inference input: {inference_data_path}')
print(f'Transform output: {eval_output_prefix}')

try:
    sm.create_transform_job(
        TransformJobName=transform_job_name,
        ModelName=model_name,
        TransformInput={
            'DataSource': {'S3DataSource': {'S3DataType': 'S3Prefix', 'S3Uri': inference_data_path}},
            'ContentType': 'text/csv',
            'SplitType': 'Line',
        },
        TransformOutput={
            'S3OutputPath': eval_output_prefix,
            'AssembleWith': 'Line',
        },
        TransformResources={'InstanceType': 'ml.m5.large', 'InstanceCount': 1},
    )
except sm.exceptions.ClientError as e:
    if 'already exists' in str(e).lower():
        print(f'Transform job {transform_job_name} already exists, using existing results')
    else:
        raise

# Wait for completion
print('Waiting for transform job to complete...')
waiter = sm.get_waiter('transform_job_completed_or_stopped')
waiter.wait(TransformJobName=transform_job_name, WaiterConfig={'Delay': 15, 'MaxAttempts': 60})

transform_info = sm.describe_transform_job(TransformJobName=transform_job_name)
assert transform_info['TransformJobStatus'] == 'Completed', f'Transform failed: {transform_info["TransformJobStatus"]}'
print(f'Transform completed: {transform_info["TransformOutput"]["S3OutputPath"]}')

In [ ]:
# Compute evaluation metrics
# Read test labels (first column of test CSV)
from urllib.parse import urlparse

parsed = urlparse(test_data_path.rstrip('/'))
bucket = parsed.netloc
# Ensure prefix ends with '/' to avoid matching test_transform/ etc.
prefix = parsed.path.lstrip('/').rstrip('/') + '/'

# List test files — only include files directly under the test/ prefix
# (not sub-prefixes like test_transform/)
test_objects = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
test_csv_files = sorted(
    [obj['Key'] for obj in test_objects.get('Contents', [])
     if obj['Key'].endswith('.csv') and obj['Key'].startswith(prefix)
     and '/' not in obj['Key'][len(prefix):]]
)

print(f'Test prefix: {prefix}')
print(f'Test CSV files found: {test_csv_files}')

labels = []
for key in test_csv_files:
    body = s3.get_object(Bucket=bucket, Key=key)['Body'].read().decode('utf-8')
    for line in body.strip().split('\n'):
        val = line.split(',')[0].strip()
        try:
            labels.append(int(float(val)))
        except ValueError:
            continue

# Read predictions from the run-specific output path
pred_parsed = urlparse(eval_output_prefix.rstrip('/'))
pred_bucket = pred_parsed.netloc
pred_prefix = pred_parsed.path.lstrip('/').rstrip('/') + '/'

pred_objects = s3.list_objects_v2(Bucket=pred_bucket, Prefix=pred_prefix)
pred_csv_out_files = sorted(
    [obj['Key'] for obj in pred_objects.get('Contents', [])
     if obj['Key'].endswith('.csv.out')]
)

print(f'Prediction prefix: {pred_prefix}')
print(f'Prediction files: {pred_csv_out_files}')

predictions = []
for key in pred_csv_out_files:
    body = s3.get_object(Bucket=pred_bucket, Key=key)['Body'].read().decode('utf-8')
    for line in body.strip().split('\n'):
        line = line.strip()
        if line:
            predictions.append(float(line))

y_true = np.array(labels)
y_prob = np.array(predictions)

# Diagnostic checks
print(f'\nLabels count: {len(y_true)}, unique values: {np.unique(y_true)}')
print(f'Predictions count: {len(y_prob)}, range: [{y_prob.min():.4f}, {y_prob.max():.4f}]')
assert len(y_true) == len(y_prob), (
    f'Mismatch: {len(y_true)} labels vs {len(y_prob)} predictions. '
    f'Test files: {test_csv_files}, Pred files: {pred_csv_out_files}'
)
assert set(np.unique(y_true)).issubset({0, 1}), (
    f'Labels must be binary (0/1), got unique values: {np.unique(y_true)}'
)

y_pred = (y_prob >= 0.5).astype(int)

# Metrics
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score

metrics = {
    'auc': roc_auc_score(y_true, y_prob),
    'accuracy': accuracy_score(y_true, y_pred),
    'f1': f1_score(y_true, y_pred),
    'precision': precision_score(y_true, y_pred),
    'recall': recall_score(y_true, y_pred),
    'test_samples': len(y_true),
}

print('\n=== Evaluation Metrics ===')
for k, v in metrics.items():
    print(f'  {k:15s}: {v:.4f}' if isinstance(v, float) else f'  {k:15s}: {v}')

In [ ]:
# Save evaluation report
evaluation_report = {
    'training_job': training_job_name,
    'model_name': model_name,
    'model_data_url': model_data_url,
    'metrics': metrics,
}

# Upload to S3
report_key = f'{pred_prefix.rstrip("/")}/evaluation_report.json'
s3.put_object(
    Bucket=pred_bucket,
    Key=report_key,
    Body=json.dumps(evaluation_report, indent=2, default=str),
    ContentType='application/json',
)
print(f'\nEvaluation report saved to s3://{pred_bucket}/{report_key}')

In [ ]:
# Log to MLflow
if tracking_server_arn and experiment_name:
    import mlflow
    os.environ['MLFLOW_TRACKING_ARN'] = tracking_server_arn
    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)

    # Find parent run from training
    runs = mlflow.search_runs(
        filter_string=f"tags.pipeline_run = '{training_job_name}' and tags.pipeline_step = 'train_model'",
        max_results=1,
    )
    parent_run_id = runs.iloc[0]['run_id'] if not runs.empty else None

    with mlflow.start_run(run_name=f'evaluate-{model_name}', parent_run_id=parent_run_id):
        mlflow.set_tag('pipeline_step', 'evaluate_model')
        mlflow.set_tag('pipeline_run', training_job_name)
        mlflow.log_metrics({k: v for k, v in metrics.items() if isinstance(v, float)})
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('test_samples', metrics['test_samples'])
        print(f'Logged evaluation metrics to MLflow')
else:
    print('MLflow not configured, skipping')

In [ ]:
# Summary
print('\n' + '=' * 60)
print('EVALUATION SUMMARY')
print('=' * 60)
print(f'  Training Job:  {training_job_name}')
print(f'  Model:         {model_name}')
print(f'  AUC:           {metrics["auc"]:.4f}')
print(f'  Accuracy:      {metrics["accuracy"]:.4f}')
print(f'  F1:            {metrics["f1"]:.4f}')
print(f'  Test Samples:  {metrics["test_samples"]}')
print('=' * 60)